# api-ops-smoke — 样本主轨（C0 / C0.5 / C1 / C2 / C3 / C4）

- **C0**：环境隔离初始化 + `TestClient` 打 `/` · `/health`
- **C0.5**：验收已有 `documents/sample/` 索引（**默认不重建**）
- **C1**：会话 CRUD + mock `/qa` 两轮 → GET 完整 turns → DELETE → 3002
- **C2**：fixture JSONL → `/stats/qa|index|health` 结构验收
- **C3**：文档列表分页 → 已知 pmcid → 未知 id 得 `3001`
- **C4**：端到端联调（会话→问答→历史→stats→documents）+ `/docs` tags 确认

样本索引路径：`Dataset/documents/sample/`（与 `full/` 分目录）。


## C0 — 环境隔离与骨架自检

In [12]:
from __future__ import annotations

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
STAGE12 = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REPO = STAGE12.parent

# --- 环境隔离：清掉可能冲突的 app / 上游模块，避免与其它阶段 notebook 串包 ---
for name in ("config", "bootstrap", "resources", "app"):
    sys.modules.pop(name, None)
for key in list(sys.modules):
    if key == "app" or key.startswith("app."):
        sys.modules.pop(key, None)

for p in (str(REPO), str(STAGE12)):
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(STAGE12))

from app.bootstrap import bootstrap_paths
from app.bridge11 import reset_stage11_cache

reset_stage11_cache()
paths = bootstrap_paths(STAGE12)
print("repo", paths["root"])
print("stage12", paths["stage12"])
print("stage11", paths["stage11"])
print("sys.path[0]", sys.path[0])
assert Path(sys.path[0]).resolve() == paths["stage12"].resolve()

repo D:\谷歌
stage12 D:\谷歌\12 服务化与接口开发第二部分
stage11 D:\谷歌\11 服务化与接口开发第一部分
sys.path[0] D:\谷歌\12 服务化与接口开发第二部分


In [13]:
from fastapi.testclient import TestClient

from app.main import app
from app.deps import get_session_store, get_qa_logger
from app.bridge11 import load_stage11

client = TestClient(app)
r = client.get("/")
print("GET /", r.status_code, r.json())
assert r.status_code == 200 and r.json()["code"] == 0
assert r.json()["data"]["stage"] == "12-6"
assert r.json()["data"]["stats"] == "qa+index+health"
assert r.json()["data"]["documents"] == "catalog"

h = client.get("/health")
print("GET /health", h.status_code, h.json().get("code"))
assert h.status_code == 200 and h.json()["code"] == 0

s11 = load_stage11()
assert get_session_store() is s11["deps"].get_session_store()
assert get_qa_logger() is s11["deps"].get_qa_logger()
print("singletons OK (shared with stage11 /qa)")
print("MemorySessionStore", type(get_session_store()).__name__)
assert r.json()["data"].get("sessions") == "crud"
print("C0 PASS")


2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/ status=200 request_id=79f1e3cc-ef63-4be3-8434-998b7f861d50 elapsed_ms=5.3
GET / 200 {'code': 0, 'message': 'ok', 'data': {'service': 'medical-rag-api', 'version': '0.1.0', 'stage': '12-4', 'retrieval_mode': 'sample', 'pipeline_backend': 'constrained10', 'documents_mode': 'sample', 'singletons': 'stage11-deps', 'sessions': 'crud', 'stats': 'qa+index+health', 'documents': 'catalog'}, 'request_id': '79f1e3cc-ef63-4be3-8434-998b7f861d50', 'timestamp': '2026-07-28T13:17:40.062176Z'}
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/health status=200 request_id=43d09adb-885b-47a6-95cc-e54160bc1fcc elapsed_ms=1.4
GET /health 200 0
singletons OK (shared with stage11 /qa)
MemorySessionStore MemorySessionStore
C0 PASS


## C0.5 — 验收 documents/sample（默认不重建）

索引已在阶段 0 落盘。日常只 **status 校验**。
若需强制重建：将 `REBUILD_SAMPLE = True` 后重跑本格。

In [14]:
from dataset_paths import (
    CHUNKS_SAMPLE_JSONL,
    DOCUMENTS_SAMPLE_DIR,
    DOCUMENTS_SAMPLE_SQLITE,
    SLIM_JSONL,
)
from app.documents_index import build_documents_index, status

# 默认关闭：样本索引已建好；仅在需要时改为 True
REBUILD_SAMPLE = False

print("slim", SLIM_JSONL.exists(), SLIM_JSONL)
print("chunks_sample", CHUNKS_SAMPLE_JSONL.exists(), CHUNKS_SAMPLE_JSONL)
print("sample_dir", DOCUMENTS_SAMPLE_DIR)
print("sqlite", DOCUMENTS_SAMPLE_SQLITE, "exists=", DOCUMENTS_SAMPLE_SQLITE.exists())

if REBUILD_SAMPLE:
    def _cb(p):
        if p.get("phase") in {"writing", "sample_complete", "completed", "finalizing"}:
            print(
                f"  lines={p.get('processed_lines')} rows={p.get('valid_rows')} "
                f"matched={p.get('matched_sample')}/{p.get('sample_target')} "
                f"phase={p.get('phase')}"
            )

    manifest = build_documents_index(
        "sample",
        batch_size=500,
        resume=False,
        progress_cb=_cb,
    )
    print("rebuilt", manifest.get("status"), "row_count", manifest.get("row_count"))
else:
    print("REBUILD_SAMPLE=False — skip build; verifying existing index")

st = status("sample")
print("status", {k: st[k] for k in ("row_count", "completed", "sqlite", "sqlite_exists")})
assert st["sqlite_exists"], f"missing {DOCUMENTS_SAMPLE_SQLITE}"
assert st["completed"] and st["row_count"] and st["row_count"] > 0
print("C0.5 PASS — documents/sample ready for stages 1–4")

slim True D:\谷歌\Dataset\processed\oa_comm_slim.jsonl
chunks_sample True D:\谷歌\Dataset\processed\chunks_sample.jsonl
sample_dir D:\谷歌\Dataset\documents\sample
sqlite D:\谷歌\Dataset\documents\sample\documents_sample.sqlite exists= True
REBUILD_SAMPLE=False — skip build; verifying existing index
status {'row_count': 1000, 'completed': True, 'sqlite': 'D:\\谷歌\\Dataset\\documents\\sample\\documents_sample.sqlite', 'sqlite_exists': True}
C0.5 PASS — documents/sample ready for stages 1–4


## C1 — 会话管理 API

显式开桌 → mock `/qa` 两轮 → GET 见 2 条完整 turns → DELETE → 再 GET 得 `3002`。

注意：`POST /qa` 对无效 session_id **自动新建**；`GET/DELETE /sessions/{id}` 无效则 **3002**。


In [15]:
from fastapi.testclient import TestClient

from app.deps import get_qa_logger, get_rag_service, get_session_store, reset_singletons
from app.main import app, _S11  # 必须用 main 挂载时的同一套 stage11 deps

class _FakePipe:
    def __init__(self):
        self.calls = []
    def run(self, query, **kwargs):
        self.calls.append(query)
        return {
            "answer": f"A:{query[-40:]}",
            "sources": [{"index": 1, "chunk_id": "c1", "doc_id": "PMC176545"}],
            "constraint_checks": {"citation": {"ok": True}},
            "generation_metrics": {"total_time_seconds": 0.01},
            "retry_count": 0,
            "repaired": False,
        }

# 只清 lru_cache，不要 reset_stage11_cache（否则 /qa 的 Depends 与 override 不是同一函数对象）
reset_singletons()

deps11 = _S11["deps"]
cfg = _S11["config"].Stage11Config(session_ttl_seconds=3600, session_max_turns=10)
store = deps11.MemorySessionStore(cfg)
pipe = _FakePipe()
rag = deps11.RagService(cfg, pipeline=pipe, inject_history=True)
qlog = deps11.QACallLogger(cfg)

app.dependency_overrides.clear()
# sessions 路由用 stage12 deps；/qa 路由用 stage11 deps —— 两边都要盖住
app.dependency_overrides[get_session_store] = lambda: store
app.dependency_overrides[get_rag_service] = lambda: rag
app.dependency_overrides[get_qa_logger] = lambda: qlog
app.dependency_overrides[deps11.get_session_store] = lambda: store
app.dependency_overrides[deps11.get_rag_service] = lambda: rag
app.dependency_overrides[deps11.get_qa_logger] = lambda: qlog

client = TestClient(app)

sid = client.post("/api/v1/sessions").json()["data"]["session_id"]
print("created", sid)

r1 = client.post("/api/v1/qa", json={"query": "first question", "session_id": sid})
r2 = client.post("/api/v1/qa", json={"query": "second question", "session_id": sid})
assert r1.status_code == 200 and r2.status_code == 200
assert r1.json()["data"]["session_id"] == sid
assert r2.json()["data"]["session_id"] == sid
# mock 时应接近瞬时；若仍 >几秒说明 override 没命中、跑了真 pipeline
assert r1.elapsed.total_seconds() < 5, f"qa too slow ({r1.elapsed}); rag override missed?"

detail = client.get(f"/api/v1/sessions/{sid}").json()["data"]
print("detail turn_count=", detail["turn_count"])
assert detail["turn_count"] == 2
assert "answer" in detail["turns"][0] and "answer_preview" not in detail["turns"][0]
assert len(pipe.calls) == 2
assert "Conversation context" in pipe.calls[1]

assert client.delete(f"/api/v1/sessions/{sid}").json()["code"] == 0
gone = client.get(f"/api/v1/sessions/{sid}")
assert gone.status_code == 404 and gone.json()["code"] == 3002

auto = client.post("/api/v1/qa", json={"query": "x", "session_id": "bad-id"})
assert auto.json()["data"]["session_id"] != "bad-id"
assert client.get("/api/v1/sessions/bad-id").json()["code"] == 3002

app.dependency_overrides.clear()
print("C1 PASS")


2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=POST path=/api/v1/sessions status=200 request_id=11efe31d-fa6a-4180-aa6d-8ba138c4f193 elapsed_ms=1.7
created e32e6d4d-1672-4930-bfd7-d83d062a9bd8
2026-07-28 21:17:40 | INFO | med_rag_api.qa_logger | qa_call status=ok code=0 request_id=52c26ec0-d0f9-4026-8b42-2dc7b98c39df latency_ms=0.0
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=POST path=/api/v1/qa status=200 request_id=52c26ec0-d0f9-4026-8b42-2dc7b98c39df elapsed_ms=5.4
2026-07-28 21:17:40 | INFO | med_rag_api.qa_logger | qa_call status=ok code=0 request_id=4be22f75-6053-472a-a46e-5382337f9154 latency_ms=0.0
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=POST path=/api/v1/qa status=200 request_id=4be22f75-6053-472a-a46e-5382337f9154 elapsed_ms=3.6
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/sessions/e32e6d4d-1672-4930-bfd7-d83d062a9bd8 status=200 request_id=da7f86f1-a374-4ddf-a7

## C2 — 运营统计 API

用 **fixture JSONL**（覆盖 `QACallLogger.path`）验收 `/stats/qa` 聚合；`/stats/index` 区分 chunk vs 文档篇数；`/stats/health` 中 `database=skipped`。

依赖 C0 已 `import app.main`。本格 **不要** `reset_stage11_cache`。


In [16]:
import json
from pathlib import Path

from fastapi.testclient import TestClient

from app.deps import get_qa_logger
from app.main import app
from app.services.stats_service import aggregate_qa_stats

# --- fixture JSONL（不污染正式 outputs/logs）---
fixture_dir = Path("_c2_fixture_logs")
fixture_dir.mkdir(exist_ok=True)
qa_path = fixture_dir / "qa_calls.jsonl"
rows = [
    {"status": "ok", "latency_ms": 1200.0, "code": 0, "request_id": "a"},
    {"status": "ok", "latency_ms": 800.0, "code": 0, "request_id": "b"},
    {"status": "error", "latency_ms": 400.0, "code": 1001, "request_id": "c"},
]
qa_path.write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n",
    encoding="utf-8",
)

local = aggregate_qa_stats(qa_path)
print("local aggregate", local.model_dump())
assert local.total_calls == 3 and local.success_count == 2
assert abs(local.avg_latency_seconds - 0.8) < 1e-6  # (1200+800+400)/3/1000

class _FakeLog:
    path = qa_path

app.dependency_overrides[get_qa_logger] = lambda: _FakeLog()
client = TestClient(app)

qa = client.get("/api/v1/stats/qa").json()
print("GET /stats/qa", qa["data"])
assert qa["code"] == 0
assert qa["data"]["total_calls"] == 3
assert qa["data"]["success_rate"] == local.success_rate
assert abs(qa["data"]["avg_latency_seconds"] - 0.8) < 1e-6

idx = client.get("/api/v1/stats/index").json()
print("GET /stats/index", {k: idx["data"][k] for k in (
    "document_count", "chunk_count", "index_size_bytes",
    "incremental_update_count", "retrieval_mode", "documents_mode",
)})
assert idx["code"] == 0
assert idx["data"]["incremental_update_count"] == 0
doc_n, chunk_n = idx["data"]["document_count"], idx["data"]["chunk_count"]
assert doc_n is not None and chunk_n is not None
assert doc_n != chunk_n, "document_count 不得填成 chunk 数"
assert doc_n == 1000  # sample sqlite

health = client.get("/api/v1/stats/health").json()
comps = {c["name"]: c for c in health["data"]["components"]}
print("GET /stats/health", {k: v["status"] for k, v in comps.items()})
assert comps["database"]["status"] == "skipped"
assert comps["api"]["status"] == "ok"
assert comps["llm"]["status"] in {"ok", "degraded", "down"}
assert comps["vector_db"]["status"] in {"ok", "degraded", "down"}

app.dependency_overrides.clear()
print("C2 PASS")


local aggregate {'total_calls': 3, 'success_count': 2, 'failure_count': 1, 'success_rate': 0.666667, 'avg_latency_seconds': 0.8}
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/stats/qa status=200 request_id=ee5e0d6c-eaaa-4602-83d4-b03da46415ba elapsed_ms=1.5
GET /stats/qa {'total_calls': 3, 'success_count': 2, 'failure_count': 1, 'success_rate': 0.666667, 'avg_latency_seconds': 0.8}
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/stats/index status=200 request_id=2077e50a-c81b-4fcb-9296-993ddef3327b elapsed_ms=5.7
GET /stats/index {'document_count': 1000, 'chunk_count': 1267, 'index_size_bytes': 16429251, 'incremental_update_count': 0, 'retrieval_mode': 'sample', 'documents_mode': 'sample'}
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/stats/health status=200 request_id=aa1eaa71-b42b-408d-99ca-97eb57a1022c elapsed_ms=29.3
GET /stats/health {'llm': 'ok', 'vector_db': 'ok'

## C3 — 文档管理 API

只读菜单册：`doc_id` = `pmcid`。列表分页 → 已知 `PMC176545` → 未知 id 得业务码 **3001**（路由未命中不再是 3001）。


In [17]:
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)

page = client.get("/api/v1/documents", params={"page": 1, "page_size": 5}).json()
print("list", {k: page["data"][k] for k in ("total", "page", "page_size")}, "n_items", len(page["data"]["items"]))
assert page["code"] == 0
assert page["data"]["total"] == 1000
assert len(page["data"]["items"]) == 5
assert page["data"]["items"][0]["doc_id"].startswith("PMC")

hit = client.get("/api/v1/documents/PMC176545").json()
print("get PMC176545 title=", (hit["data"]["title"] or "")[:60])
assert hit["code"] == 0 and hit["data"]["doc_id"] == "PMC176545"

miss = client.get("/api/v1/documents/PMC_DOES_NOT_EXIST")
print("missing", miss.status_code, miss.json()["code"], miss.json()["message"])
assert miss.status_code == 404 and miss.json()["code"] == 3001

route_miss = client.get("/api/v1/no-such-route-xyz")
print("route miss code=", route_miss.json()["code"])
assert route_miss.status_code == 404 and route_miss.json()["code"] != 3001

print("C3 PASS")


2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/documents status=200 request_id=516a8322-baa0-411e-89e6-1755c20673e9 elapsed_ms=3.1


list {'total': 1000, 'page': 1, 'page_size': 5} n_items 5
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/documents/PMC176545 status=200 request_id=425efabc-57b0-4eea-bc0a-230fc21401a7 elapsed_ms=2.1
get PMC176545 title= The Transcriptome of the Intraerythrocytic Developmental Cyc
2026-07-28 21:17:40 | WARNING | med_rag_api.app.core.exceptions | app_exception code=3001 status=404 path=/api/v1/documents/PMC_DOES_NOT_EXIST detail={'doc_id': 'PMC_DOES_NOT_EXIST'}
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/documents/PMC_DOES_NOT_EXIST status=404 request_id=852c7335-45ab-4eae-afd9-4c253c23098e elapsed_ms=5.4
missing 404 3001 document not found
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/api/v1/no-such-route-xyz status=404 request_id=66d0a09c-a8cc-4ecc-ae4c-d4e036662537 elapsed_ms=0.5
route miss code= 1001
C3 PASS


## C4 — 样本全链路 + OpenAPI

创建会话 → mock `/qa` 两轮 → 查历史 → stats → documents（对照 QA 的 pmcid）→ 确认 `/openapi.json` tags 含 sessions/stats/documents。

本格复用 C1 的 override 思路（`_S11` + 双端 Depends）；**不要** `reset_stage11_cache`。


In [18]:
from fastapi.testclient import TestClient

from app.deps import get_qa_logger, get_rag_service, get_session_store, reset_singletons
from app.main import app, _S11

class _FakePipe:
    def __init__(self):
        self.calls = []
    def run(self, query, **kwargs):
        self.calls.append(query)
        return {
            "answer": f"A:{query[-40:]}",
            "sources": [{"index": 1, "chunk_id": "c1", "doc_id": "PMC176545"}],
            "constraint_checks": {"citation": {"ok": True}},
            "generation_metrics": {"total_time_seconds": 0.01},
            "retry_count": 0,
            "repaired": False,
        }

reset_singletons()
deps11 = _S11["deps"]
cfg = _S11["config"].Stage11Config(session_ttl_seconds=3600, session_max_turns=10)
store = deps11.MemorySessionStore(cfg)
pipe = _FakePipe()
rag = deps11.RagService(cfg, pipeline=pipe, inject_history=True)
qlog = deps11.QACallLogger(cfg)

app.dependency_overrides.clear()
app.dependency_overrides[get_session_store] = lambda: store
app.dependency_overrides[get_rag_service] = lambda: rag
app.dependency_overrides[get_qa_logger] = lambda: qlog
app.dependency_overrides[deps11.get_session_store] = lambda: store
app.dependency_overrides[deps11.get_rag_service] = lambda: rag
app.dependency_overrides[deps11.get_qa_logger] = lambda: qlog

client = TestClient(app)

assert client.get("/").json()["data"]["stage"] == "12-6"

sid = client.post("/api/v1/sessions").json()["data"]["session_id"]
r1 = client.post("/api/v1/qa", json={"query": "C4 first", "session_id": sid})
r2 = client.post("/api/v1/qa", json={"query": "C4 second", "session_id": sid})
assert r1.status_code == 200 and r2.status_code == 200

hist = client.get(f"/api/v1/sessions/{sid}").json()["data"]
print("history turns", hist["turn_count"])
assert hist["turn_count"] == 2

qa_stats = client.get("/api/v1/stats/qa").json()["data"]
idx = client.get("/api/v1/stats/index").json()["data"]
print("stats/qa total_calls", qa_stats["total_calls"], "index docs", idx["document_count"])
assert idx["document_count"] == 1000

pmcid = r1.json()["data"]["sources"][0]["doc_id"]
doc = client.get(f"/api/v1/documents/{pmcid}").json()
print("document", pmcid, "→", doc["code"], (doc["data"].get("title") or "")[:50])
assert doc["code"] == 0 and doc["data"]["doc_id"] == pmcid

schema = client.get("/openapi.json").json()
tags = {t["name"] for t in schema.get("tags", [])}
print("openapi tags", sorted(tags))
for name in ("health", "qa", "sessions", "stats", "documents"):
    assert name in tags
assert client.get("/docs").status_code == 200

app.dependency_overrides.clear()
print("C4 PASS — sample e2e + OpenAPI tags OK")
print("部署说明见 docs/部署与API调用说明.md；Postman: postman/MedRAG_API.postman_collection.json")


2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=GET path=/ status=200 request_id=64aee3ac-d230-4597-b906-e4c08861588d elapsed_ms=1.2
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=POST path=/api/v1/sessions status=200 request_id=0bf65ca5-ef3a-45e6-804a-1de0706b3238 elapsed_ms=1.7
2026-07-28 21:17:40 | INFO | med_rag_api.qa_logger | qa_call status=ok code=0 request_id=9888a2cc-1c0e-4aeb-94ca-1c9e8e015b20 latency_ms=0.0
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=POST path=/api/v1/qa status=200 request_id=9888a2cc-1c0e-4aeb-94ca-1c9e8e015b20 elapsed_ms=4.4
2026-07-28 21:17:40 | INFO | med_rag_api.qa_logger | qa_call status=ok code=0 request_id=c4001ad8-cdd5-4a6c-b559-f13359e4fb55 latency_ms=0.0
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request method=POST path=/api/v1/qa status=200 request_id=c4001ad8-cdd5-4a6c-b559-f13359e4fb55 elapsed_ms=4.0
2026-07-28 21:17:40 | INFO | med_rag_api.middleware | request meth